In [4]:
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import mannwhitneyu, chi2_contingency

df = pd.read_csv("Train_Data.csv")
TARGET = "Dropped_Course"
baseline_rate = df[TARGET].mean()
print(df.shape)
print(f"overall {TARGET} rate: {baseline_rate:.4f}")

ModuleNotFoundError: No module named 'pandas'

# Tier 1 — behavioral / history variables

## Prev_Course_Dropouts

In [ ]:
col = "Prev_Course_Dropouts"
print(df[col].describe())

plt.hist(df[col], bins=range(0, df[col].max() + 2))
plt.title(f"{col} distribution")
plt.xlabel(col)
plt.show()

rate_by_val = df.groupby(col)[TARGET].agg(["mean", "count"])
print(rate_by_val)

dropped = df.loc[df[TARGET] == 1, col]
stayed = df.loc[df[TARGET] == 0, col]
stat, p = mannwhitneyu(dropped, stayed)
print(f"\nMann-Whitney U p-value: {p:.3e}")

**What can you infer about `Prev_Course_Dropouts`?**

*(write interpretation here)*

## Returning_Client

In [ ]:
col = "Returning_Client"
print(df[col].value_counts())

rate_by_val = df.groupby(col)[TARGET].agg(["mean", "count"])
print(rate_by_val)

rate_by_val["mean"].plot(kind="bar")
plt.axhline(baseline_rate, color="red", linestyle="--", label="baseline")
plt.title(f"{TARGET} rate by {col}")
plt.legend()
plt.show()

table = pd.crosstab(df[col], df[TARGET])
chi2, p, dof, _ = chi2_contingency(table)
print(f"chi2={chi2:.2f}, p={p:.3e}")

**What can you infer about `Returning_Client`?**

*(write interpretation here)*

## Prev_Course_Attended

In [ ]:
col = "Prev_Course_Attended"
print(df[col].describe())

plt.hist(df[col], bins=30)
plt.title(f"{col} distribution")
plt.xlabel(col)
plt.show()

dropped = df.loc[df[TARGET] == 1, col]
stayed = df.loc[df[TARGET] == 0, col]

plt.boxplot([stayed, dropped], labels=["stayed", "dropped"])
plt.title(f"{col} by {TARGET}")
plt.show()

stat, p = mannwhitneyu(dropped, stayed)
print(f"Mann-Whitney U p-value: {p:.3e}")

**What can you infer about `Prev_Course_Attended`?**

*(write interpretation here)*

## Registration_Days_Before

In [ ]:
col = "Registration_Days_Before"
print(df[col].describe())
print("missing:", df[col].isna().sum(), f"({df[col].isna().mean():.2%})")

plt.hist(df[col].dropna(), bins=50)
plt.title(f"{col} distribution")
plt.xlabel(col)
plt.show()

dropped = df.loc[df[TARGET] == 1, col].dropna()
stayed = df.loc[df[TARGET] == 0, col].dropna()

plt.boxplot([stayed, dropped], labels=["stayed", "dropped"])
plt.title(f"{col} by {TARGET}")
plt.show()

stat, p = mannwhitneyu(dropped, stayed)
print(f"Mann-Whitney U p-value: {p:.3e}")

# is missingness itself related to the target?
print(df.groupby(df[col].isna())[TARGET].mean())

**What can you infer about `Registration_Days_Before`?**

*(write interpretation here)*

## Waiting_List_Days

In [ ]:
col = "Waiting_List_Days"
print(df[col].describe())

plt.hist(df[col], bins=50)
plt.title(f"{col} distribution")
plt.xlabel(col)
plt.show()

dropped = df.loc[df[TARGET] == 1, col]
stayed = df.loc[df[TARGET] == 0, col]

plt.boxplot([stayed, dropped], labels=["stayed", "dropped"])
plt.title(f"{col} by {TARGET}")
plt.show()

stat, p = mannwhitneyu(dropped, stayed)
print(f"Mann-Whitney U p-value: {p:.3e}")

**What can you infer about `Waiting_List_Days`?**

*(write interpretation here)*

## Registration_Changes

In [ ]:
col = "Registration_Changes"
print(df[col].describe())

plt.hist(df[col], bins=range(0, df[col].max() + 2))
plt.title(f"{col} distribution")
plt.xlabel(col)
plt.show()

rate_by_val = df.groupby(col)[TARGET].agg(["mean", "count"])
print(rate_by_val)

dropped = df.loc[df[TARGET] == 1, col]
stayed = df.loc[df[TARGET] == 0, col]
stat, p = mannwhitneyu(dropped, stayed)
print(f"\nMann-Whitney U p-value: {p:.3e}")

**What can you infer about `Registration_Changes`?**

*(write interpretation here)*

## Pre_Course_Supports_Tickets

In [ ]:
col = "Pre_Course_Supports_Tickets"
print(df[col].describe())

plt.hist(df[col], bins=range(0, df[col].max() + 2))
plt.title(f"{col} distribution")
plt.xlabel(col)
plt.show()

rate_by_val = df.groupby(col)[TARGET].agg(["mean", "count"])
print(rate_by_val)

dropped = df.loc[df[TARGET] == 1, col]
stayed = df.loc[df[TARGET] == 0, col]
stat, p = mannwhitneyu(dropped, stayed)
print(f"\nMann-Whitney U p-value: {p:.3e}")

**What can you infer about `Pre_Course_Supports_Tickets`?**

*(write interpretation here)*

# Tier 2 — dirty categoricals (text cleaning needed)

These columns have inflated cardinality from stray characters, whitespace, and case variants (e.g. `" PRT"`, `"PRT#"`, `"prt"` are all the same value). Clean inline before analyzing.

## Origin_Country

In [ ]:
col = "Origin_Country"
raw = df[col].astype(str)
cleaned = raw.str.upper().str.replace(r"[^A-Z]", "", regex=True)
cleaned = cleaned.replace("CN", "CHN")
cleaned = cleaned.where(df[col].notna(), "MISSING")
df["country_clean"] = cleaned

print("raw unique:", df[col].nunique(), "-> cleaned unique:", df["country_clean"].nunique())

vc = df["country_clean"].value_counts()
print(vc.head(20))

top = vc.head(15).index
rate_by_country = df[df["country_clean"].isin(top)].groupby("country_clean")[TARGET].agg(["mean", "count"]).sort_values("count", ascending=False)
print(rate_by_country)

rate_by_country["mean"].plot(kind="bar")
plt.axhline(baseline_rate, color="red", linestyle="--", label="baseline")
plt.title(f"{TARGET} rate by country (top 15)")
plt.legend()
plt.show()

table = pd.crosstab(df["country_clean"], df[TARGET])
chi2, p, dof, _ = chi2_contingency(table)
print(f"chi2={chi2:.2f}, dof={dof}, p={p:.3e}")

**What can you infer about `Origin_Country`?**

*(write interpretation here)*

## Next variable

Copy the pattern above for the remaining Tier 2/3/4 variables (see `stage1_plan.md` for the full order).

In [ ]:
col = "NEXT_COLUMN"